In [1]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import pathlib
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

E0000 00:00:1750181879.607561   36692 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750181879.611904   36692 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750181879.626271   36692 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750181879.626301   36692 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750181879.626303   36692 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750181879.626305   36692 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# import project defined modules / functions used in this notebook
# ensure that the src directory where project modules are found is on
# the PYTHONPATH
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from nndl import vectorize_samples, plot_history

In [3]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
dev = tf.config.list_physical_devices()
print('Physical Devices : ', dev)

#tf.config.set_visible_devices(dev[0])
#tf.config.set_visible_devices(dev[1])
dev = tf.config.list_logical_devices()
print('Available Devices : ', dev)

Physical Devices :  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Available Devices :  [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:GPU:0', device_type='GPU')]


I0000 00:00:1750181881.905714   36692 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9706 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


# Chapter 11: Deep Learning Text

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

In this section we look at the particular types of deep network architectures that work well when processing textual time series, as
well as other aspects specific to preparing and processing textual input.

## 11.3 Two Approaches for Representing Groups of Words: Sets and sequences

- Simplest approach: discard order and treat as unordered set: **bag-of-words models**
- Process strictly in order they appear, like steps in a timeseries **sequence models**
- Hybrid approach: **Transformer architecture** technically order-agnostic, yet injects word-postiion info into representations.

In this section we'll return to the IMDB movie reviews dataset.  We'll demonstrate each approach (bag-of-words and sequence models) on
this dataset and see how they do.

### 11.3.1 Preparing the IMDB movie reviews data

Though we are using the same dataset, for practice on the datapreprocessing and doing vectorization, we will get
the raw dataset and approach it as a new text-classification problem.

You can download the dataset from the following url.  If using our class DevContainers, the following expects the dataset to be
untared into the `../data/aclImdb/` directory, using the following commands. The `.tar.gz` file containing the data here is 81M in size,
and it extracts to a size of 433M total: 

```bash
vscode ➜ /workspaces/nndl/data (main) $ curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  7852k      0  0:00:10  0:00:10 --:--:-- 11.1M

vscode ➜ /workspaces/nndl/data (main) $ tar -xf aclImdb_v1.tar.gz 

vscode ➜ /workspaces/nndl/data (main) $ du -h -s aclImdb
433M    aclImdb

# we don't need the unsup data in train, get rid of it for space
vscode ➜ /workspaces/nndl/data (main) $ rm -rf aclImdb/train/unsup
```

This directory has a subdirectory structure you should be familair with:

```bash
vscode ➜ /workspaces/nndl/data (main) $ tree -d aclImdb/
aclImdb/
├── test
│   ├── neg
│   └── pos
└── train
    ├── neg
    └── pos
```


That is to say there is a corpus of textual reviews split into training and testing data.  This is a binary classification task, as you may recall, with an
even split of positive and negative reviews.  For instance the `train/pos/` subdirectory contains a set of 12,500 (plain ascii) text files, each of
which contains the text body of a positive-sentiment movie review.

It would be informative to look at a few reviews:

```bash
vscode ➜ /workspaces/nndl/data (main) $ cat aclImdb/train/pos/4077_10.txt 
I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drink and food in reach so you don't have to stop the film.<br /><br />
```

Let's prepare a validation set by setting apart 20% of the training text files in a new directory, aclImdb/val:

In [4]:
base_dir = pathlib.Path("../data/aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"

# if the val_dir exists, assume that this code has already run and don't select validation set again
# WARNING: the files are actually moved from train to validation.  So running this multiple times
# is not something you want to do.
if not os.path.exists(val_dir):
    for category in ("neg", "pos"):
        print(f"selecting {category} validation reviews: ", end='')
        # make sure the destination directory is there for the copying
        os.makedirs(val_dir / category)
        # get list of all current training files
        files = os.listdir(train_dir / category)
    
        # shuffle the list of training files using a seed, to ensure we get the same validation
        # set every time we run the code.
        random.Random(1337).shuffle(files)
        # selection 20% to copy over for validation purposes
        num_val_samples = int(0.2 * len(files))
        val_files = files[-num_val_samples:]
        
        for fname in val_files:
            shutil.move(train_dir / category / fname, val_dir / category / fname)
            print('.', end='')
        print('')

As hinted at, we have a similar subdirectory structure, so we can use a similar utility method from keras for streaming text
datasets called `text_dataset_from_directory`.  Let's create three `Dataset` objects for training, validation and testing:

**Note**: good idea to verify you get 20,000 files from train, 5,000 from validation after split and full 25,000 still in test datasets here.

In [5]:
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    train_dir, batch_size=batch_size
)

val_ds = keras.utils.text_dataset_from_directory(
    val_dir, batch_size=batch_size
)

test_ds = keras.utils.text_dataset_from_directory(
    test_dir, batch_size=batch_size
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


These datasets yield inputs that are TensorFlow `tf.string` tensors, and targets are `int32` tensors encoding values of
"0" or "1".

In [6]:
for inputs, targets in train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

inputs.shape: (32,)
inputs.dtype: <dtype: 'string'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor(b"I was hardly aware of the time in history depicted in this 1971 Brazilian black comedy, however that is not to say it wasn't accessible to me because the movie makes it very clear. It's set in 16th century Brazil, where rival French and Portuguese settlers are exploiting the indigineous people as confederates in their battle to assert dominance. What is particularly interesting about the movie is that it is made by the Portuguese from the point of view of the French. The hero is a likable Frenchman, the Portuguese are barbarians, and the rest of the French are oppressive and greedy. The film's Portuguese makers are objective because when all is said and done, we see that it makes no difference whose side one takes. It's about heredity overpowered by environment in a time starkly defined by tribes. Enemies are made and perpetuated, and like so, the environmenta

The inputs have not been tokenized or vectorized yet, they are just long strings here.

Notice that the 0'th input is labeled 0.  I think that because `neg` subdirectory name comes pefore `pos` subdirectory name, that we get the usual
encoding of negative reviews to 0 and positivie to 1.  I suspect that the `keras.utils` to stream from directories have an option to map subdirectory
names to desired label, or at a minimum you could always rename your directorys like `0-neg`, `1-pos` so they end up in order you want the
target labels to be assigned.

### 11.3.2 Processing words as a set: The bag-of-words approach

The simplest way to encode a piece of text for processing by a machine learning
model is to discard order and treat it as a set (a "bag") of tokens.  

You could either look at individual words (unigrams) or try to recover some local order information
by looking at groups of consecutive tokens (N-gram).

**Note**: You may recall that in our first example using this IMDB pos/neg binary classification that we multi-hot encoded the input texts into a vector of 10,000
features.  This was basically a unigram encoding into a set (the word was present or not present somewhere in the review).

#### Single words (unigrams) with binary encoding

If we use a bag of single words, the sentence "the cat sat on the mat" becomes

```
{"cat", "mat", "on", "sat", "the"}
```

The main advantage of this encoding is that you can represent an entire text as a single
vector, where each entry is a presence indicator for a given word. For instance,
using binary encoding (multi-hot), you’d encode a text as a vector with as many
dimensions as there are words in your vocabulary—with 0s almost everywhere and
some 1s for dimensions that encode words present in the text. This is what we did
when we worked with text data in chapters 4 and 5.

Let's process our raw text datasets with a `TextVectorization` layer so that they
yield multi-hot encoded binary word vectors.  Our layer will only look at 
single words (that is to say, **unigrams**).

In [7]:
# limit vocabulary to 20,000 most frequent words, twice more than last time
# In general, 20,000 is about the right vocabulary size for text classification on
# a real world corpus
#
# also notice the parameter to specify encoding as multi-hot binary vectors
text_vectorization = layers.TextVectorization(
    max_tokens=20000,
    output_mode="multi_hot",
)

# prepare a dataset that only yields raw text inputs (no labels)
text_only_train_ds = train_ds.map(lambda x, y: x)

# use that dataset to index the dataset vocabulary via the adapt() method,
# so like before this initializes our text_vectorization instance with the training 
# corpus vocabulary
text_vectorization.adapt(text_only_train_ds)

# prepare processed versions of training, validation and test dataset
# e.g. these datasets produce binary encoded vectors of 1-gram input
# make sure to specify num_parallel_calls to leverage multi CPU cores
binary_1gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_1gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_1gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

Let's inspect the resulting TensorFlow `Dataset` to again make sure we understood the preprocessing that has just taken place.

If you recall, the `Dataset` class can be treated as an iterator, so we can ask it to give us the first batch of inputs
and labels like this.  Do you know what we should be expecting from iterating over this dataset?:

In [8]:
for inputs, targets in binary_1gram_train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

inputs.shape: (32, 20000)
inputs.dtype: <dtype: 'int64'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor([1 1 1 ... 0 0 0], shape=(20000,), dtype=int64)
targets[0]: tf.Tensor(0, shape=(), dtype=int32)


We set the batch size to 32.  And the mapping we did in the previous cell causes the vectorization to happen, so that the
variably sized text reviews are mapped as 1-grams into a 20000 shaped vector with a 1 at each loacation where the
corresponding word index for the word in the review appears.

Next let's write a reusable model-building function that we'll use in all of our experiments in this section.

In [9]:
def get_model(max_tokens=20000, hidden_dim=16):
    """
    """
    # a simple single dense layer with dropout model
    # our task is a binary classification, so final layer has 1 output
    # using sigmoid activation
    inputs = keras.Input(shape=(max_tokens,))
    x = layers.Dense(hidden_dim, activation="relu")(inputs)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    # create and compile the model for binary classification
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

Finally let's train and test this first model.

In [10]:
model = get_model()
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-binary-1gram.keras", save_best_only=True)
]

# notice the call to cache() for the dataset, this will cache them in primary RAM memory.
# This way we will do the preprocessing to vectorize only 1 time during epoch 1, and will reuse the
# preprocessed texts for the following epochs.  This can only be done when data is small enough to
# fit into memory
history = model.fit(binary_1gram_train_ds.cache(),
                    validation_data=binary_1gram_val_ds.cache(),
                    epochs=10,
                    callbacks=callbacks)

# reload best seen model by validation loss to test
model = keras.models.load_model("../models/ch11-binary-1gram.keras")
print(f"Test acc: {model.evaluate(binary_1gram_test_ds)[1]:.3f}")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


I0000 00:00:1750181944.089522   36801 service.cc:152] XLA service 0x7a2b8004f6e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750181944.089614   36801 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
I0000 00:00:1750181944.232854   36801 cuda_dnn.cc:529] Loaded cuDNN version 90300


  9/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5560 - loss: 0.6938  

I0000 00:00:1750181944.728520   36801 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.7689 - loss: 0.4904 - val_accuracy: 0.8750 - val_loss: 0.3080
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8912 - loss: 0.2780 - val_accuracy: 0.8850 - val_loss: 0.3050
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9113 - loss: 0.2385 - val_accuracy: 0.8868 - val_loss: 0.3226
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9232 - loss: 0.2238 - val_accuracy: 0.8802 - val_loss: 0.3496
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9244 - loss: 0.2207 - val_accuracy: 0.8810 - val_loss: 0.3637
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9286 - loss: 0.2176 - val_accuracy: 0.8818 - val_loss: 0.3667
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9319 - loss: 0.2120 - val_accuracy: 0.8848 - val_loss: 0.3793
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9326 - loss: 0.2114 - val_accuracy: 0.8788 - va

This should usually achieve a test accuracy of around 89%.  Not bad.  If you go and look back, the first time we did this on the
IMDB database with 2 layers of 16 units we got about 87% accuracy.  Though there we used only the first 10,000
words instead of 20,000 in a multi-hot encoding, and it may have been overfitting, which might be why we get a bit
better performance here usually with only a single layer.

Note that in this case the dataset is a balanced two-class 
classification dataset, so the naive baseline we could reach without training would only be 50%. Meanwhile
just FYI, the best score that can be achieved on this dataset without leveraging external data is around 95% test accuracy.

#### Bigrams with binary encoding

Of course discarding word order is very reductive, because even atomic concepts can be
expressed via multiple word terms: "United States" conveys a single concept that is differented
from "united" and "states".

For this reason, you will usually end up re-injecting local order information int a bag-of-words representation
by looking at N-grams rather than single words.

The `TextVectorization` layer can be configured to return arbitrary N-grams: bigrams, trigrams, etc.
Just pass an `ngrams=N` argument:

**Note**: It looks like the `keras.layers.TextVectorization` does return 1 and 2-gram by default when using
its `ngrams` parameter, not just the 2-grams.

In [11]:
# create the TextVectorization instance again, but use
# 2-grams for the vocabulary
text_vectorization = layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="multi_hot",
)

# we create the 2-gram vocabulary here
text_vectorization.adapt(text_only_train_ds)

binary_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

Just to make sure we are clear here, what does the vocabulary look like now in the
`text_vectorization` instance once we trained on 2-gram input:

In [12]:
vocabulary = text_vectorization.get_vocabulary()
print(len(vocabulary))
print(vocabulary[:100])

20000
['[UNK]', 'the', 'and', 'a', 'of', 'to', 'is', 'in', 'it', 'i', 'this', 'that', 'br', 'was', 'as', 'for', 'with', 'movie', 'but', 'of the', 'film', 'on', 'not', 'you', 'are', 'his', 'have', 'be', 'he', 'one', 'its', 'in the', 'at', 'all', 'by', 'an', 'they', 'from', 'who', 'so', 'like', 'her', 'or', 'just', 'about', 'if', 'has', 'out', 'some', 'there', 'this movie', 'what', 'good', 'more', 'when', 'very', 'and the', 'is a', 'even', 'my', 'no', 'she', 'would', 'up', 'the film', 'to the', 'which', 'to be', 'time', 'really', 'only', 'story', 'their', 'see', 'had', 'were', 'the movie', 'can', 'this film', 'me', 'than', 'it is', 'we', 'much', 'this is', 'well', 'get', 'will', 'been', 'other', 'also', 'because', 'do', 'into', 'bad', 'great', 'people', 'on the', 'in a', 'how']


You should see if you look closely that there are 2-word pairs along with the single words in the dictionary.
Also you might want to consider, the number of possible 2-grams has extended the size of the potential
input vocabularly considerably.  I wonder if increasing the `max_tokens` would help here or not.

Let's test how our model performs when trained on such binary-encoded bags of bigrams.  Note because
`max_tokens` is still set to 20,000, the output from our `Dataset`s will still be a multi-hot
vector of 20,000 features.

In [13]:
model = get_model()
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-binary-2gram.keras", save_best_only=True)
]


model.fit(binary_2gram_train_ds.cache(),
          validation_data=binary_2gram_val_ds.cache(),
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model("../models/ch11-binary-2gram.keras")
print(f"Test acc: {model.evaluate(binary_2gram_test_ds)[1]:.3f}")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.7892 - loss: 0.4691 - val_accuracy: 0.8880 - val_loss: 0.2810
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9080 - loss: 0.2432 - val_accuracy: 0.8894 - val_loss: 0.2938
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9337 - loss: 0.2035 - val_accuracy: 0.8890 - val_loss: 0.3134
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9397 - loss: 0.1854 - val_accuracy: 0.8894 - val_loss: 0.3311
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9463 - loss: 0.1791 - val_accuracy: 0.8864 - val_loss: 0.3575
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9488 - loss: 0.1648 - val_accuracy: 0.8852 - val_loss: 0.3811
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9539 - loss: 0.1614 - val_accuracy: 0.8874 - val_loss: 0.3960
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9536 - loss: 0.1587 - val_accuracy: 

You should now usually get at or above 90% on test accuracy, which is a definite improvement.  Turns out local
order is pretty important.

#### Bigrams with TF-IDF encoding

You can add a bit more information to this representation by counting how many times each word or 
N-gram occurs in an input sample.  That is to say, by taking histograms of words over the text.

If you are doing text classification, knowing how many times a word occurs in a sample is critical:
any sufficiently long movie review may contain the word "terrible" regardless of sentiment, but a review
that contains many instances of "terrible" is likely a negative one.

Our previous multi-encoding to vectorize the input that we did by hand could easily be modified
to encode the counts of words/bigrams, simply add 1 to the value in the dictionary each time we
see it when encoding.

For the `TextVectorization` layer, simply use the `output_mode="count"` parameter.

In [14]:
text_vectorization = layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="count"
)

Another topic we haven't addressed yet, normalization of texts when doing text processing
tasks.  Small common words like "the", "a", "is" in English will always dominate your
word count histograms, drowning out other words, despite being pretty much
uselss features in a classificaiton context like this.

We could normalize word counts by subtracting the mean and dividing by the variance. Except most
vectorized sentences consist almost entirely of zeros, a property called sparsity.
Using the basic normalization (mean centering and dividing by std) would wreck the sparsity,
which for computational reasons you would like to preserve.

Instead we use **TF-IDF normalization**, which stand for "term frequency inverse document frequency".

TF-IDF is so common that it's built into the `TextVectorization` layer.  All you need to do to
start using it is to switch the `output_mode` to "tf_idf".

In [15]:
# have the TextVectorization perform the TF-IDF normalization on token histograms in samples
text_vectorization = layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="tf_idf",
)

# the adapt() call will learn the TF-IDF weights in addition
# to the vocabulary
text_vectorization.adapt(text_only_train_ds)

tfidf_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

tfidf_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

tfidf_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [16]:
model = get_model()
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-tfidf-2gram.keras", save_best_only=True)
]

history = model.fit(tfidf_2gram_train_ds.cache(),
                    validation_data=tfidf_2gram_val_ds.cache(),
                    epochs=10,
                    callbacks=callbacks)

model = keras.models.load_model("../models/ch11-tfidf-2gram.keras")
print(f"Test acc: {model.evaluate(tfidf_2gram_test_ds)[1]:.3f}")

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.7108 - loss: 0.5882 - val_accuracy: 0.8874 - val_loss: 0.2935
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8665 - loss: 0.3329 - val_accuracy: 0.8860 - val_loss: 0.2946
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8932 - loss: 0.2725 - val_accuracy: 0.8776 - val_loss: 0.3607
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8981 - loss: 0.2617 - val_accuracy: 0.8826 - val_loss: 0.3337
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9058 - loss: 0.2389 - val_accuracy: 0.8864 - val_loss: 0.3529
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9117 - loss: 0.2278 - val_accuracy: 0.8766 - val_loss: 0.3444
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9078 - loss: 0.2284 - val_accuracy: 0.8786 - val_loss: 0.3500
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9144 - loss: 0.2084 - val_accuracy: 

For the IMDB dataset, you probably won't see any improvement here using TF-IDF.  However for
many text-classification datasets, it would be typical to see a performance increase when
using TF-IDF compared to the plain histogram counts.

#### Exporting a model that processes raw strings

In the preceding examples, we did our text standardization, splitting, and indexing as
part of the tf.data pipeline. But if we want to export a standalone model independent
of this pipeline, we should make sure that it incorporates its own text preprocessing
(otherwise, you’d have to reimplement in the production environment, which
can be challenging or can lead to subtle discrepancies between the training data and
the production data). Thankfully, this is easy in Keras.

Just create a new model that reuses your TextVectorization layer and adds to it
the model you just trained:

In [17]:
# notice change in Input here, a vector of only 1 element, which is a string data type
inputs = keras.Input(shape=(1,), dtype="string")
processed_inputs = text_vectorization(inputs)
outputs = model(processed_inputs)

inference_model = keras.Model(inputs, outputs)

# the new model has the trained text_vectorization instance in it, before feeding into
# the previous trained model, which expects exactly the output that the text_vectorizaiton does
inference_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_3            │ (None, 20000)          │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ functional_2 (Functional)       │ (None, 1)              │       320,033 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

The resulting model can process batches of raw strings.

In [18]:
raw_text_data = tf.convert_to_tensor([
    ["That was an excellent movie, I loved it."],
    ["Awful, do not waste your money. Avoid seeing it."],
])

predictions = inference_model(raw_text_data)
print(f"{float(predictions[0] * 100):.2f} percent positive")
print(f"{float(predictions[1] * 100):.2f} percent positive")

89.11 percent positive
6.73 percent positive


## Summary

<font color='blue'>
    
- Word order in Text processing is handled in 2 basic ways:
  1. **bag-of-words models** : discard order and treat as an unordered set (multi-hot encoding typically, 20,000 sparse vectors).
  2. **sequence models**: process words in order they appear like a timeseries
- For sequence models, can encode sequence again using one-hot encoding, but this ends up with very large input to RNN models.
- **word embeddings** are vector representations of words that map humanlanguage into a structured geometric space.